In [97]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [98]:
files = [
    "CRMLSSold202511.csv",
    "CRMLSSold202512.csv",
    "CRMLSSold202602.csv",
    "CRMLSSold202603.csv",
    "CRMLSSold202604.csv",
    "CRMLSSold202605.csv"
]

dfs = [pd.read_csv(file) for file in files]

df = pd.concat(dfs, ignore_index=True)

In [99]:
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
]

### Inspect Missing Values

In [100]:
print(len(df))
print("\nMissing values after preprocessing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

63976

Missing values after preprocessing:
BuyerAgentAOR                    3663
ListAgentAOR                       17
Flooring                        23159
ViewYN                           5652
WaterfrontYN                    63945
                                ...  
HighSchoolDistrict              17233
PostalCode                          1
AssociationFee                  18466
LotSizeSquareFeet                1079
MiddleOrJuniorSchoolDistrict    63976
Length: 61, dtype: int64


In [101]:
# check how many missing values each boolean column has
cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

print(df[cols].isna().sum())

ViewYN               5652
PoolPrivateYN        5102
AttachedGarageYN     7733
FireplaceYN            55
NewConstructionYN    4850
dtype: int64



### Handling missing values








In [102]:
# Drop columns where more than 70% of values are missing
threshold = 0.7
df = df.loc[:, df.isnull().mean() < threshold]

In [103]:
# drop rows that have missing values in the ClosePrice column
df = df.dropna(subset=["ClosePrice"])

In [104]:
# flag and impute the AttachedGarageYN column (the only boolean column with missing values)
df["AttachedGarageYN_missing"] = df["AttachedGarageYN"].isna().astype(int)

df["AttachedGarageYN"] = df["AttachedGarageYN"].fillna(df["AttachedGarageYN"].mode()[0])

/tmp/ipykernel_9414/3989187805.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["AttachedGarageYN"] = df["AttachedGarageYN"].fillna(df["AttachedGarageYN"].mode()[0])


In [105]:
# force key numeric columns into numbers
# then, fill missing values with the Median
core_features = ["LivingArea", "LotSizeArea", "BedroomsTotal", "BathroomsTotalInteger"]
for col in core_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())


In [106]:
# replace missing values in the City and Flooring columns with "Unknown"
df["City"] = df["City"].fillna("Unknown")
df["Flooring"] = df["Flooring"].fillna("Unknown")

### Convert categorical fields to numeric (encoding)

In [107]:
df = pd.get_dummies(df, columns=["City"], drop_first=True)

In [108]:

df = pd.get_dummies(df, columns=["Flooring"], drop_first=True)


### Create train/test split

In [109]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df["Month"] = df["CloseDate"].dt.to_period("M")

months = sorted(df["Month"].unique())

X = 3

test_month = months[-1]
train_months = months[-(X + 1):-1]

train_df = df[df["Month"].isin(train_months)].copy()
test_df = df[df["Month"] == test_month].copy()

In [110]:
num_cols = ["LivingArea", "LotSizeArea", "BedroomsTotal", "BathroomsTotalInteger"]

In [111]:
# fit scaler on train data only
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df[num_cols] = scaler.fit_transform(train_df[num_cols])

In [112]:
test_df[num_cols] = scaler.transform(test_df[num_cols])

### cleaned CSV:

In [113]:
df.to_csv("cleaned_dataset.csv", index=False)